# Einstein summation

[Einstein notation](https://en.wikipedia.org/wiki/Einstein_notation) is a convention for simplifying equations with many summations over matrices and vectors – or tensors in general, introduced to physics (from Ricci) by Einstein for his work on general relativity. It makes the summation sign implicit whenever there are repeated indices. For example, the $i$ th entry of the product $Ab$ of a matrix $A$ and a vector $b$ can be written
$$
(Ab)_i = \sum_{j} A_{ij} b_j = A^i_j b^j .
$$
Originally (and in general relativity texts), the indices are written as subscripts and superscripts like here, indicating elements of rows and columns, respectively, but this is just for clarity. We could also just write $A_{ij}b_j$, in which case it looks more like ordinary tensor index notation.

[NumPy implements the Einstein summation](https://numpy.org/doc/stable/reference/generated/numpy.einsum.html) as `einsum`, and this is a rather powerful tool when you work with vectors, matrices, tensors or multi-dimensional arrays in general. All common matrix operations like multiplication, inner product, transpose, trace, etc. can be calculated with `einsum` (not that you necessarily should do it), but it is really for more complex applications you'd typically use it: it can do things that are hard to do with other matrix functions, and it can increase readability for heavy matrix manipulations. 

It can be hard to grok how to use it, but fortunately Lev Maximov (of the Illustrated NumPy tutorial) wrote another great visual guide focused on `einsum` alone: [Einsum Visualized](https://medium.com/better-programming/einsum-visualized-c050903145ef). Also the [Einstein Summation in NumPy](https://obilaniu6266h16.wordpress.com/2016/02/04/einstein-summation-in-numpy/) tutorial by Olexa Bilaniuk is enlightening, including an argument for why to use it:
> [...] it can reduce the number of errors made by computer scientists and reduce the time they spend reasoning about linear algebra. It does so by being simultaneously clearer, more explicit, more self-documenting, more declarative in style and less cognitively burdensome to use.

Below are some basic examples.

In [1]:
import numpy as np

A = np.array([[0, 1], [2, 3]])
b = np.array([4, 5])

print("A:\n", A)
print("b:\n", b)

A:
 [[0 1]
 [2 3]]
b:
 [4 5]


In [2]:
# trace of A
print(np.einsum('ii', A),
      A.trace())

3 3


In [3]:
# no repeated indices - nothing happens
np.einsum('ij', A)

array([[0, 1],
       [2, 3]])

In [4]:
# inner product of b with itself
print(
    np.einsum('i,i', b, b),
    b @ b,
    np.inner(b, b))

41 41 41


In [6]:
# matrix product of A and b
print(
    np.einsum('ij,j', A, b),
    A @ b,
    np.dot(A, b)
)

[ 5 23] [ 5 23] [ 5 23]


In [ ]:
# A^2
print( np.einsum('ij, jk', A, A) )
print( A @ A )
print( np.dot(A, A) )

All of the above make use of the _implicit_ output index convention – that the indices appear in the output in alphabetical order. That means we can take the matrix transpose like this (because we named the input indices in reverse alphabetical order, they'll be swapped in the output):

In [8]:
print( np.einsum('ji', A) )
print( A.T )

[[0 2]
 [1 3]]
[[0 2]
 [1 3]]


It is, however, possible to explicitly specify the output indices using `->`, which opens up for many new applications of the function beyond the original Einstein notation. Examples:

In [9]:
# reverse order is maintained, so nothing happens to A
np.einsum('ji->ji', A)

array([[0, 1],
       [2, 3]])

In [10]:
# A^T A
print( np.einsum('ij,ik->jk', A, A) )
print( A.T @ A )


[[ 4  6]
 [ 6 10]]
[[ 4  6]
 [ 6 10]]


Indices not appearing in the output index specification are summed over.\
Indices appearing once in the input and once in the output are left alone (see the [Five rules of Einsum](https://medium.com/better-programming/einsum-visualized-c050903145ef#a514) section):

In [11]:
# summation over just one axis
print( np.einsum('ij->i', A) )
print( A.sum(axis=1) )

print( np.einsum('ij->j', A) )
print( A.sum(axis=0) )

[1 5]
[1 5]
[2 4]
[2 4]


In [12]:
# summing all elements
print( np.einsum('i->', b) )
print( b.sum() )

print( np.einsum('ij->', A) )
print( A.sum() )

9
9
6
6


We can calculate the effect of applying 30 $R_x(\phi) = e^{-i\phi\hat{X}}$ gates for $\phi$ varying from 0 to $2\pi$ to 5 different states located on the arc between $|0\rangle$ and $|+\rangle$, both included.\
The output should be a 30 × 5 array of qubits (2-long vector), i.e. an array of shape (30, 5, 2).

In [13]:
from scipy.linalg import expm

# set up the 30 gates
phis = np.linspace(0, 2*np.pi, 30, endpoint=False)
X = np.array([[0, 1], [1, 0]])
gates = expm(-1j * phis[:, None, None] * X / 2)

# set up the 5 states
thetas = np.linspace(0, np.pi/2, 5)
states = np.array([np.cos(thetas / 2), np.sin(thetas / 2)]).T

print(gates.shape, states.shape)

(30, 2, 2) (5, 2)


In [14]:
# apply the gates
outputs = np.einsum('ijk,lk->ilj', gates, states)

In [15]:
np.set_printoptions(precision=3, suppress=True)
print(outputs.shape)
print(outputs)

(30, 5, 2)
[[[ 1.   +0.j     0.   +0.j   ]
  [ 0.981+0.j     0.195+0.j   ]
  [ 0.924+0.j     0.383+0.j   ]
  [ 0.831+0.j     0.556+0.j   ]
  [ 0.707+0.j     0.707+0.j   ]]

 [[ 0.995+0.j     0.   -0.105j]
  [ 0.975-0.02j   0.194-0.103j]
  [ 0.919-0.04j   0.381-0.097j]
  [ 0.827-0.058j  0.553-0.087j]
  [ 0.703-0.074j  0.703-0.074j]]

 [[ 0.978+0.j     0.   -0.208j]
  [ 0.959-0.041j  0.191-0.204j]
  [ 0.904-0.08j   0.374-0.192j]
  [ 0.813-0.116j  0.543-0.173j]
  [ 0.692-0.147j  0.692-0.147j]]

 [[ 0.951+0.j     0.   -0.309j]
  [ 0.933-0.06j   0.186-0.303j]
  [ 0.879-0.118j  0.364-0.285j]
  [ 0.791-0.172j  0.528-0.257j]
  [ 0.672-0.219j  0.672-0.219j]]

 [[ 0.914+0.j     0.   -0.407j]
  [ 0.896-0.079j  0.178-0.399j]
  [ 0.844-0.156j  0.35 -0.376j]
  [ 0.76 -0.226j  0.508-0.338j]
  [ 0.646-0.288j  0.646-0.288j]]

 [[ 0.866+0.j     0.   -0.5j  ]
  [ 0.849-0.098j  0.169-0.49j ]
  [ 0.8  -0.191j  0.331-0.462j]
  [ 0.72 -0.278j  0.481-0.416j]
  [ 0.612-0.354j  0.612-0.354j]]

 [[ 0.809+0.j    